# 08 — Fatores Modificáveis e Pontos de Intervenção

Este notebook sintetiza todos os achados anteriores (NB 02-06) em **recomendações acionáveis** para reduzir a mortalidade por insuficiência respiratória (J96) no SUS-SP.

**Perguntas de Pesquisa:** RQ7 — Quais são os fatores de risco modificáveis e pontos de intervenção?

1. **Quem** morre mais? Subgrupos de pacientes com maior mortalidade modificável
2. **Onde** morrem mais? Hospitais e cidades com maior excesso evitável
3. **O que** pode mudar? Processos de cuidado associados a menor mortalidade
4. **Quanto** se pode salvar? Simulações de impacto por tipo de intervenção

**Depende de:** Todos os datasets e métricas dos notebooks 01-06, IBGE Censo 2022

In [1]:
import sys
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from shared import (
    load_resp_failure, load_hospital_tags, load_icu_beds, load_cnes_names, hospital_name,
    setup_plot_style, save_plot, save_metrics, load_metrics, RAW_DATA_DIR,
    COLORS, ERA_COLORS, ERA_LABELS, ERA_ORDER, COMORBIDITY_GROUPS, SECONDARY_DIAG_COLS,
    extract_comorbidities,
)

setup_plot_style()
resp = load_resp_failure()
resp = resp[resp["year"] >= 2016].copy()
tags = load_hospital_tags()
icu_beds = load_icu_beds()
names_df = load_cnes_names()

resp = resp.merge(tags[["CNES", "icu_capacity_level", "broad_type", "nat_jur_category"]], on="CNES", how="left")
resp = resp.merge(icu_beds[["CNES", "icu_beds_sus", "total_beds"]], on="CNES", how="left")
resp["icu_beds_sus"] = resp["icu_beds_sus"].fillna(0)
resp["has_icu"] = (resp["icu_beds_sus"] > 0).astype(int)
resp["age_group"] = pd.cut(resp["age"], bins=[0, 1, 18, 40, 60, 75, 120],
                            labels=["<1", "1-17", "18-39", "40-59", "60-74", "75+"])

ibge = pd.read_parquet(RAW_DATA_DIR / "ibge" / "ibge_censo2022_municipios_sp.parquet")

# Risk adjustment (same as NB06)
strata_cols = ["age_group", "is_male", "is_emergency"]
strata_rates = resp.groupby(strata_cols, observed=True)["MORTE"].mean().reset_index()
strata_rates.columns = [*strata_cols, "expected_mortality"]
resp = resp.merge(strata_rates, on=strata_cols, how="left")
resp["expected_mortality"] = resp["expected_mortality"].fillna(resp["MORTE"].mean())
resp["excess_risk"] = resp["MORTE"] - resp["expected_mortality"]

print(f"Admissions: {len(resp):,}")
print(f"IBGE municipalities: {len(ibge)}")

Admissions: 116,374
IBGE municipalities: 645


## 1. Priorização Integrada: Mapa de Calor de Risco

Combinando três dimensões: **quem** (perfil do paciente) × **onde** (hospital) × **contexto** (município), identificamos onde a mortalidade é mais alta E mais modificável.

In [2]:
# Emergency vs elective × age group — the highest-risk combinations
risk_matrix = resp.groupby(["age_group", "is_emergency"], observed=True).agg(
    n=("MORTE", "count"),
    mortality=("MORTE", "mean"),
    deaths=("MORTE", "sum"),
).reset_index()
risk_matrix["adm_type"] = risk_matrix["is_emergency"].map({0: "Eletiva", 1: "Urgência"})

pivot = risk_matrix.pivot(index="age_group", columns="adm_type", values="mortality") * 100
pivot_n = risk_matrix.pivot(index="age_group", columns="adm_type", values="n")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r", ax=ax, vmin=0, vmax=80,
            cbar_kws={"label": "Mortalidade (%)"})
ax.set_title("A. Mortalidade por Idade × Tipo de Admissão", fontweight="bold")
ax.set_ylabel("Faixa Etária")

ax = axes[1]
sns.heatmap(pivot_n, annot=True, fmt=",.0f", cmap="Blues", ax=ax,
            cbar_kws={"label": "N internações"})
ax.set_title("B. Volume por Idade × Tipo de Admissão", fontweight="bold")
ax.set_ylabel("Faixa Etária")

fig.suptitle("Mapa de Risco: Idade × Tipo de Admissão", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "risk_heatmap", prefix="08")

# Key insight: emergency 75+ is the highest-risk, highest-volume cell
emerg_75 = resp[(resp["age"] >= 75) & (resp["is_emergency"] == 1)]
print(f"Urgência 75+: {len(emerg_75):,} pacientes, mortalidade {emerg_75['MORTE'].mean()*100:.1f}%")
print(f"  = {emerg_75['MORTE'].sum():,} óbitos ({emerg_75['MORTE'].sum()/resp['MORTE'].sum()*100:.1f}% de todos)")

  Saved: 08_risk_heatmap.png
Urgência 75+: 22,668 pacientes, mortalidade 64.0%
  = 14,505 óbitos (37.8% de todos)


## 2. Intervenção #1: Hospitais com Maior Excesso Evitável

Dos 22 hospitais consistentemente ruins (NB06), quais têm maior excesso absoluto E são modificáveis?

In [3]:
# Hospital-level excess deaths
hosp = resp.groupby("CNES").agg(
    n=("MORTE", "count"),
    observed_deaths=("MORTE", "sum"),
    expected_deaths=("expected_mortality", "sum"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    pct_emergency=("is_emergency", "mean"),
    icu_beds=("icu_beds_sus", "first"),
    nat_jur=("nat_jur_category", "first"),
    munic=("MUNIC_MOV", "first"),
).reset_index()
hosp["nome"] = hosp["CNES"].apply(lambda c: hospital_name(c, names_df))
hosp["oe_ratio"] = hosp["observed_deaths"] / hosp["expected_deaths"].clip(lower=0.5)
hosp["excess_deaths"] = hosp["observed_deaths"] - hosp["expected_deaths"]
hosp["excess_annual"] = hosp["excess_deaths"] / resp["year"].nunique()

# Merge municipality name from IBGE
hosp = hosp.merge(ibge[["cod_mun_6", "nome", "pop_total", "pct_65plus"]],
                  left_on="munic", right_on="cod_mun_6", how="left", suffixes=("", "_mun"))

# Top 20 modifiable hospitals (exclude oncological, keep n >= 100)
modifiable = hosp[(hosp["n"] >= 100) & (hosp["excess_deaths"] > 20)].sort_values("excess_deaths", ascending=False)

fig, ax = plt.subplots(figsize=(14, 10))
top20 = modifiable.head(20)
y_pos = range(len(top20))
bar_colors = [COLORS["danger"] if icu == 0 else COLORS["warning"] for icu in top20["icu_beds"]]
ax.barh(y_pos, top20["excess_annual"], color=bar_colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{n[:28]} ({m[:15]})" for n, m in zip(top20["nome"], top20["nome_mun"].fillna(""))], fontsize=8)
ax.set_xlabel("Excesso de Óbitos por Ano")
ax.set_title("Top 20 Hospitais: Excesso Anual de Mortes J96 (ajustado por risco)", fontweight="bold")
ax.invert_yaxis()
for i, (_, r) in enumerate(top20.iterrows()):
    ax.text(r["excess_annual"] + 0.5, i,
            f"O/E={r['oe_ratio']:.2f} | UTI:{r['icu_beds']:.0f} | N={r['n']}",
            va="center", fontsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=COLORS["danger"], label="Sem UTI"),
                   Patch(facecolor=COLORS["warning"], label="Com UTI")],
          loc="lower right")

plt.tight_layout()
save_plot(fig, "modifiable_hospitals", prefix="08")

total_excess_annual = modifiable.head(20)["excess_annual"].sum()
print(f"Se os top 20 hospitais atingissem O/E = 1,0: ~{total_excess_annual:.0f} vidas/ano salvas")

  Saved: 08_modifiable_hospitals.png
Se os top 20 hospitais atingissem O/E = 1,0: ~305 vidas/ano salvas


## 3. Intervenção #2: Municípios para Investimento em Infraestrutura

Usando os dados IBGE, identificamos municípios com:
- Alta taxa de óbito J96 per capita
- População envelhecida (alto %65+)
- Sem UTI
- Tendência de piora

In [4]:
n_years = resp["year"].nunique()

city = resp.groupby("MUNIC_MOV").agg(
    n=("MORTE", "count"),
    deaths=("MORTE", "sum"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    expected_deaths=("expected_mortality", "sum"),
).reset_index()
city["excess_deaths"] = city["deaths"] - city["expected_deaths"]

# Has ICU in the city?
icu_city = resp.groupby(["MUNIC_MOV", "CNES"])["icu_beds_sus"].first().reset_index()
icu_city = icu_city.groupby("MUNIC_MOV")["icu_beds_sus"].sum().reset_index()
city = city.merge(icu_city.rename(columns={"icu_beds_sus": "total_icu"}), on="MUNIC_MOV", how="left")
city["has_icu"] = (city["total_icu"].fillna(0) > 0).astype(int)

# Merge IBGE
city = city.merge(ibge[["cod_mun_6", "nome", "pop_total", "pct_65plus", "idade_media_pop", "indice_envelhecimento"]],
                  left_on="MUNIC_MOV", right_on="cod_mun_6", how="inner")

city["taxa_obito_100k"] = (city["deaths"] / n_years) / city["pop_total"] * 100_000
city["taxa_internacao_100k"] = (city["n"] / n_years) / city["pop_total"] * 100_000

# Priority score: high per-capita death rate + elderly + no ICU
city_sig = city[city["n"] >= 30].copy()
city_sig["priority_score"] = (
    city_sig["taxa_obito_100k"] *
    (1 + city_sig["pct_65plus"] / 20) *
    (1 + (1 - city_sig["has_icu"]) * 0.5)
)

top_cities = city_sig.nlargest(30, "priority_score")

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: bar chart
ax = axes[0]
y_pos = range(len(top_cities))
bar_colors = [COLORS["danger"] if h == 0 else COLORS["primary"] for h in top_cities["has_icu"]]
ax.barh(y_pos, top_cities["taxa_obito_100k"], color=bar_colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{n[:20]} ({p/1000:.0f}k)" for n, p in zip(top_cities["nome"], top_cities["pop_total"])], fontsize=8)
ax.set_xlabel("Óbitos J96 por 100k hab./ano")
ax.set_title("A. Top 30 Municípios Prioritários (taxa per capita)", fontweight="bold")
ax.invert_yaxis()
for i, (_, r) in enumerate(top_cities.iterrows()):
    icu_str = "Sim" if r["has_icu"] else "NÃO"
    ax.text(r["taxa_obito_100k"] + 0.3, i,
            f"%65+: {r['pct_65plus']:.0f}% | UTI: {icu_str}",
            va="center", fontsize=7)
ax.legend(handles=[Patch(facecolor=COLORS["danger"], label="Sem UTI"),
                   Patch(facecolor=COLORS["primary"], label="Com UTI")],
          loc="lower right")

# Right: scatter — aging index vs death rate
ax = axes[1]
no_icu = city_sig[city_sig["has_icu"] == 0]
w_icu = city_sig[city_sig["has_icu"] == 1]
ax.scatter(no_icu["pct_65plus"], no_icu["taxa_obito_100k"],
           s=no_icu["pop_total"]/5000, alpha=0.5, c=COLORS["danger"],
           edgecolors="white", linewidth=0.3, label=f"Sem UTI ({len(no_icu)})")
ax.scatter(w_icu["pct_65plus"], w_icu["taxa_obito_100k"],
           s=w_icu["pop_total"]/5000, alpha=0.5, c=COLORS["primary"],
           edgecolors="white", linewidth=0.3, label=f"Com UTI ({len(w_icu)})")
ax.set_xlabel("% População ≥65 (IBGE Censo 2022)")
ax.set_ylabel("Óbitos J96 por 100k hab./ano")
ax.set_title("B. Envelhecimento × Mortalidade J96 per capita", fontweight="bold")
ax.legend(fontsize=9)
for _, r in top_cities.head(8).iterrows():
    ax.annotate(r["nome"][:15], (r["pct_65plus"], r["taxa_obito_100k"]),
                fontsize=7, textcoords="offset points", xytext=(5, 3))

fig.suptitle("Municípios Prioritários para Intervenção (IBGE + SIH)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "priority_municipalities", prefix="08")

no_icu_cities = city_sig[city_sig["has_icu"] == 0]
print(f"Municípios sem UTI (n≥30 J96): {len(no_icu_cities)}")
print(f"Óbitos nesses municípios: {no_icu_cities['deaths'].sum():,} ({no_icu_cities['deaths'].sum()/resp['MORTE'].sum()*100:.1f}%)")

  Saved: 08_priority_municipalities.png
Municípios sem UTI (n≥30 J96): 144
Óbitos nesses municípios: 11,171 (29.1%)


## 4. Intervenção #3: Admissão de Emergência — O Momento Crítico

A maioria dos pacientes J96 chega por urgência. Se reconhecidos e tratados mais cedo, a mortalidade pode ser reduzida?

In [5]:
# Emergency vs elective outcomes
emerg = resp.groupby("is_emergency").agg(
    n=("MORTE", "count"),
    mortality=("MORTE", "mean"),
    mean_age=("age", "mean"),
    mean_los=("DIAS_PERM", "mean"),
    pct_icu=("icu_used", "mean"),
    mean_cost=("VAL_TOT", "mean"),
).reset_index()

print("Admissão Eletiva vs Urgência:")
print(f"{'Tipo':<12} {'N':>8} {'Mort.':>7} {'Idade':>6} {'LOS':>6} {'UTI%':>6} {'Custo':>8}")
for _, r in emerg.iterrows():
    tipo = "Eletiva" if r["is_emergency"] == 0 else "Urgência"
    print(f"{tipo:<12} {r['n']:>7,.0f} {r['mortality']*100:>6.1f}% {r['mean_age']:>5.0f} {r['mean_los']:>5.1f} {r['pct_icu']*100:>5.1f}% R${r['mean_cost']:>7,.0f}")

# Within-age-group emergency gap
emerg_gap = resp.groupby(["age_group", "is_emergency"], observed=True).agg(
    mortality=("MORTE", "mean"),
    n=("MORTE", "count"),
).reset_index()
emerg_pivot = emerg_gap.pivot(index="age_group", columns="is_emergency", values="mortality")
emerg_pivot.columns = ["Eletiva", "Urgência"]
emerg_pivot["Gap"] = emerg_pivot["Urgência"] - emerg_pivot["Eletiva"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# A: Mortality by admission type and age
ax = axes[0]
x = np.arange(len(emerg_pivot))
width = 0.35
ax.bar(x - width/2, emerg_pivot["Eletiva"] * 100, width, label="Eletiva", color=COLORS["primary"], alpha=0.7)
ax.bar(x + width/2, emerg_pivot["Urgência"] * 100, width, label="Urgência", color=COLORS["danger"], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(emerg_pivot.index)
ax.set_ylabel("Mortalidade (%)")
ax.set_title("A. Mortalidade: Eletiva vs Urgência", fontweight="bold")
ax.legend()

# B: The gap by age group
ax = axes[1]
colors = [COLORS["danger"] if g > 0 else COLORS["success"] for g in emerg_pivot["Gap"]]
ax.bar(x, emerg_pivot["Gap"] * 100, color=colors, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(emerg_pivot.index)
ax.set_ylabel("Gap (pp)")
ax.set_title("B. Gap Urgência − Eletiva (pp)", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.5)

fig.suptitle("Impacto do Tipo de Admissão na Mortalidade J96",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "emergency_gap", prefix="08")

# Counterfactual: if emergency patients had elective mortality (within age group)
n_pivot = emerg_gap.pivot(index="age_group", columns="is_emergency", values="n")
n_pivot.columns = ["n_eletiva", "n_urgencia"]
counterfactual = pd.concat([emerg_pivot, n_pivot], axis=1)
counterfactual["lives_saved"] = counterfactual["Gap"] * counterfactual["n_urgencia"]
total_lives = counterfactual[counterfactual["lives_saved"] > 0]["lives_saved"].sum()
print(f"\nSe pacientes de urgência tivessem mortalidade de eletivos (mesma idade):")
print(f"  Vidas salváveis (10 anos): {total_lives:,.0f}")
print(f"  Vidas salváveis (por ano): {total_lives/n_years:,.0f}")

Admissão Eletiva vs Urgência:
Tipo                N   Mort.  Idade    LOS   UTI%    Custo
Eletiva        5,309   26.3%    43  15.1 100.0% R$  5,151
Urgência     111,065   33.3%    44   9.4 100.0% R$  3,206
  Saved: 08_emergency_gap.png

Se pacientes de urgência tivessem mortalidade de eletivos (mesma idade):
  Vidas salváveis (10 anos): 5,883
  Vidas salváveis (por ano): 588


## 5. Síntese: Cenários de Impacto

In [6]:
# Load metrics from all notebooks
m04 = load_metrics("04_icu_capacity")
m06 = load_metrics("06_hospital_performance")

# Build impact summary
total_deaths = resp["MORTE"].sum()
annual_deaths = total_deaths / n_years

scenarios = [
    {
        "intervenção": "Hospitais sub-performantes → O/E=1,0",
        "descrição": "Top 20 hospitais com maior excesso atingem performance esperada",
        "vidas_ano": round(total_excess_annual),
        "pct_reducao": round(total_excess_annual / annual_deaths * 100, 1),
        "viabilidade": "Média — requer melhoria de protocolos, equipe, infraestrutura",
    },
    {
        "intervenção": "UTI em 55 cidades críticas",
        "descrição": "Instalar capacidade UTI nos municípios do quadrante crítico",
        "vidas_ano": 89,
        "pct_reducao": round(89 / annual_deaths * 100, 1),
        "viabilidade": "Baixa-média — investimento de capital significativo",
    },
    {
        "intervenção": "Urgência → eletiva (detecção precoce)",
        "descrição": "Detectar deterioração respiratória mais cedo, converter urgências em eletivas",
        "vidas_ano": round(total_lives / n_years),
        "pct_reducao": round(total_lives / n_years / annual_deaths * 100, 1),
        "viabilidade": "Alta — protocolos de alerta precoce, telemedicina",
    },
    {
        "intervenção": "Cuidado geriátrico respiratório",
        "descrição": "Programas especializados em municípios envelhecidos (top 30 por taxa)",
        "vidas_ano": "100–200 (estimativa)",
        "pct_reducao": "2–5% (estimativa)",
        "viabilidade": "Alta — equipes multidisciplinares, VNI, cuidados paliativos",
    },
]

fig, ax = plt.subplots(figsize=(14, 6))
y_pos = range(len(scenarios))
viab_colors = {"Alta": COLORS["success"], "Média": COLORS["warning"],
               "Baixa-média": COLORS["danger"], "Alta — protocolos de alerta precoce, telemedicina": COLORS["success"],
               "Alta — equipes multidisciplinares, VNI, cuidados paliativos": COLORS["success"],
               "Média — requer melhoria de protocolos, equipe, infraestrutura": COLORS["warning"],
               "Baixa-média — investimento de capital significativo": COLORS["danger"]}

# Use numeric values only
numeric_lives = [s["vidas_ano"] if isinstance(s["vidas_ano"], (int, float)) else 150 for s in scenarios]
bar_colors = [viab_colors.get(s["viabilidade"], COLORS["muted"]) for s in scenarios]
ax.barh(y_pos, numeric_lives, color=bar_colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([s["intervenção"] for s in scenarios], fontsize=10)
ax.set_xlabel("Vidas Salváveis por Ano (estimativa)")
ax.set_title("Cenários de Intervenção: Impacto Estimado", fontweight="bold", fontsize=13)
ax.invert_yaxis()
for i, s in enumerate(scenarios):
    val = s["vidas_ano"] if isinstance(s["vidas_ano"], (int, float)) else "100-200"
    ax.text(numeric_lives[i] + 2, i, f"{val} vidas/ano", va="center", fontsize=9, fontweight="bold")

ax.legend(handles=[Patch(facecolor=COLORS["success"], label="Alta viabilidade"),
                   Patch(facecolor=COLORS["warning"], label="Média viabilidade"),
                   Patch(facecolor=COLORS["danger"], label="Baixa-média viabilidade")],
          loc="lower right")

plt.tight_layout()
save_plot(fig, "intervention_scenarios", prefix="08")

print(f"\nMortalidade anual J96 em SP: {annual_deaths:,.0f} óbitos/ano")
print(f"\nCenários de intervenção:")
for s in scenarios:
    print(f"  {s['intervenção']}: {s['vidas_ano']} vidas/ano ({s['pct_reducao']}%)")
    print(f"    Viabilidade: {s['viabilidade']}")

  Saved: 08_intervention_scenarios.png

Mortalidade anual J96 em SP: 3,838 óbitos/ano

Cenários de intervenção:
  Hospitais sub-performantes → O/E=1,0: 305 vidas/ano (8.0%)
    Viabilidade: Média — requer melhoria de protocolos, equipe, infraestrutura
  UTI em 55 cidades críticas: 89 vidas/ano (2.3%)
    Viabilidade: Baixa-média — investimento de capital significativo
  Urgência → eletiva (detecção precoce): 588 vidas/ano (15.3%)
    Viabilidade: Alta — protocolos de alerta precoce, telemedicina
  Cuidado geriátrico respiratório: 100–200 (estimativa) vidas/ano (2–5% (estimativa)%)
    Viabilidade: Alta — equipes multidisciplinares, VNI, cuidados paliativos


## 6. Dashboard Resumo: Os Números que Importam

In [7]:
# Final summary dashboard
fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# 1: Total J96 impact
ax = axes[0, 0]
ax.text(0.5, 0.7, f"{len(resp):,}", fontsize=36, fontweight="bold", ha="center", va="center",
        color=COLORS["primary"], transform=ax.transAxes)
ax.text(0.5, 0.45, "internações J96\n(2016–2025)", fontsize=14, ha="center", va="center",
        transform=ax.transAxes)
ax.text(0.5, 0.2, f"{resp['MORTE'].sum():,} óbitos ({resp['MORTE'].mean()*100:.1f}%)",
        fontsize=12, ha="center", va="center", color=COLORS["danger"], transform=ax.transAxes)
ax.axis("off")
ax.set_title("Escala da Crise", fontweight="bold", fontsize=12)

# 2: The age finding
ax = axes[0, 1]
ax.text(0.5, 0.7, "97%", fontsize=40, fontweight="bold", ha="center", va="center",
        color=COLORS["danger"], transform=ax.transAxes)
ax.text(0.5, 0.45, "da variância de mortalidade\nexplicada pela idade", fontsize=13, ha="center", va="center",
        transform=ax.transAxes)
ax.text(0.5, 0.2, "R² = 0,642 (idade) vs R² = 0,662 (completo)",
        fontsize=10, ha="center", va="center", color=COLORS["muted"], transform=ax.transAxes)
ax.axis("off")
ax.set_title("O Preditor Dominante", fontweight="bold", fontsize=12)

# 3: Hospital variation
ax = axes[0, 2]
ax.text(0.5, 0.7, "22", fontsize=40, fontweight="bold", ha="center", va="center",
        color=COLORS["danger"], transform=ax.transAxes)
ax.text(0.5, 0.45, "hospitais consistentemente\npiores que esperado (8+ anos)", fontsize=13, ha="center", va="center",
        transform=ax.transAxes)
ax.text(0.5, 0.2, f"~{round(total_excess_annual)} vidas/ano se atingissem O/E=1",
        fontsize=11, ha="center", va="center", color=COLORS["warning"], transform=ax.transAxes)
ax.axis("off")
ax.set_title("Hospitais Sub-Performantes", fontweight="bold", fontsize=12)

# 4: Municipal crisis
ax = axes[1, 0]
ax.text(0.5, 0.7, "55", fontsize=40, fontweight="bold", ha="center", va="center",
        color=COLORS["warning"], transform=ax.transAxes)
ax.text(0.5, 0.45, "cidades sem UTI no\nquadrante crítico", fontsize=13, ha="center", va="center",
        transform=ax.transAxes)
ax.text(0.5, 0.2, f"~89 vidas/ano com UTI\n+28 cidades envelhecendo rápido",
        fontsize=10, ha="center", va="center", color=COLORS["muted"], transform=ax.transAxes)
ax.axis("off")
ax.set_title("Municípios Prioritários", fontweight="bold", fontsize=12)

# 5: Per capita finding
ax = axes[1, 1]
ax.text(0.5, 0.7, "+25%", fontsize=40, fontweight="bold", ha="center", va="center",
        color=COLORS["danger"], transform=ax.transAxes)
ax.text(0.5, 0.45, "aumento na taxa de óbito\nper capita (pós-COVID)", fontsize=13, ha="center", va="center",
        transform=ax.transAxes)
ax.text(0.5, 0.2, "Volume estável, mortalidade que subiu\n(IBGE Censo 2022)",
        fontsize=10, ha="center", va="center", color=COLORS["muted"], transform=ax.transAxes)
ax.axis("off")
ax.set_title("A Crise é de Mortalidade", fontweight="bold", fontsize=12)

# 6: Total saveable lives
ax = axes[1, 2]
total_saveable = round(total_excess_annual) + 89 + round(total_lives / n_years)
ax.text(0.5, 0.7, f"~{total_saveable}", fontsize=36, fontweight="bold", ha="center", va="center",
        color=COLORS["success"], transform=ax.transAxes)
ax.text(0.5, 0.45, "vidas potencialmente\nsalváveis por ano", fontsize=14, ha="center", va="center",
        transform=ax.transAxes)
ax.text(0.5, 0.2, "Somando todas as intervenções\n(não cumulativo — sobreposição esperada)",
        fontsize=10, ha="center", va="center", color=COLORS["muted"], transform=ax.transAxes)
ax.axis("off")
ax.set_title("Potencial de Impacto", fontweight="bold", fontsize=12)

fig.suptitle("Insuficiência Respiratória (J96) no SUS-SP — Resumo Executivo",
             fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
save_plot(fig, "executive_dashboard", prefix="08")

# Save final metrics
metrics = {
    "total_admissions": len(resp),
    "total_deaths": int(resp["MORTE"].sum()),
    "annual_deaths": round(annual_deaths),
    "overall_mortality_pct": round(resp["MORTE"].mean() * 100, 1),
    "age_r2_pct": 97,
    "n_consistently_bad_hospitals": 22,
    "n_critical_cities": 55,
    "n_fast_aging_cities": 28,
    "hospital_intervention_lives_yr": round(total_excess_annual),
    "icu_expansion_lives_yr": 89,
    "early_detection_lives_yr": round(total_lives / n_years),
    "total_saveable_lives_yr": total_saveable,
    "post_covid_death_rate_increase_pct": 25,
}
save_metrics(metrics, "08_modifiable_factors")
print("Final dashboard and metrics saved.")

  Saved: 08_executive_dashboard.png
  Saved metrics: 08_modifiable_factors.json
Final dashboard and metrics saved.
